# Experiment Results

Four comparison tables across two axes:
- **Tables 1 & 2** — Round #1 vs Round #2 (data volume/quality effect)
- **Tables 3 & 4** — Round #2 vs ROI (preprocessing effect: padded 384×384 vs raw variable size)

Tables 1 & 3 use unbalanced training sets. Tables 2 & 4 use balanced training sets.

All per-class accuracy values are from the best validation checkpoint (selected by AUC).

In [1]:
import json
import re
import pandas as pd
from pathlib import Path

ROOT = Path("../..")
EXP_DIR = ROOT / "experiments"
META_DIR = ROOT / "data" / "metadata"

EXP_RE = re.compile(
    r"^(cnn|cytofm_head|cytofm_finetune)"
    r"_(annotated_old|annotated|roi)"
    r"_(nosus|sus)"
    r"(_balanced)?"
    r"_\d{8}_\d{6}$"
)

MODEL_LABELS = {
    "cnn": "Baseline CNN",
    "cytofm_head": "CytoFM (head only)",
    "cytofm_finetune": "CytoFM (fine-tune)",
}
DATASET_LABELS = {
    "annotated_old": "Round #1",
    "annotated": "Round #2",
    "roi": "ROI",
}

# Dataset sizes (total files, all splits)
DATASET_SIZES = {
    ("annotated_old", "nosus"): len(pd.read_csv(META_DIR / "metadata_old_nosus.csv")),
    ("annotated_old", "sus"):   len(pd.read_csv(META_DIR / "metadata_old.csv")),
    ("annotated",     "nosus"): len(pd.read_csv(META_DIR / "metadata.csv")),
    ("annotated",     "sus"):   len(pd.read_csv(META_DIR / "metadata_full.csv")),
    ("roi",           "nosus"): len(pd.read_csv(META_DIR / "metadata_roi_nosus.csv")),
    ("roi",           "sus"):   len(pd.read_csv(META_DIR / "metadata_roi_sus.csv")),
}

In [2]:
def load_results():
    """Scan experiments/ and return a dict keyed by (model, dataset, sus, balanced).
    When multiple runs exist for the same config, keeps the most recent."""
    results = {}
    for exp_path in sorted(EXP_DIR.iterdir()):
        if exp_path.name == "archive":
            continue
        m = EXP_RE.match(exp_path.name)
        if not m:
            continue
        model, dataset, sus, balanced = m.group(1), m.group(2), m.group(3), bool(m.group(4))
        history_path = exp_path / "metrics" / "training_history.json"
        if not history_path.exists():
            continue
        with open(history_path) as f:
            h = json.load(f)
        key = (model, dataset, sus, balanced)
        # keep most recent (sorted dirs give chronological order)
        results[key] = {
            "auc":       h.get("best_val_auc", None),
            "acc":       h.get("best_val_acc", None),
            "per_class": h.get("best_val_per_class_acc", []),
            "exp_dir":   exp_path.name,
        }
    return results

results = load_results()
print(f"Loaded {len(results)} experiments")
for k in sorted(results):
    r = results[k]
    pc = r['per_class']
    print(f"  {str(k):<60}  AUC={r['auc']:.3f}  B={pc[0]:.3f}  M={pc[1]:.3f}" if pc else
          f"  {str(k):<60}  AUC={r['auc']:.3f}  (no per-class)")

Loaded 36 experiments
  ('cnn', 'annotated', 'nosus', False)                          AUC=0.596  B=0.391  M=0.775
  ('cnn', 'annotated', 'nosus', True)                           AUC=0.562  B=0.350  M=0.774
  ('cnn', 'annotated', 'sus', False)                            AUC=0.631  B=0.515  M=0.672
  ('cnn', 'annotated', 'sus', True)                             AUC=0.610  B=0.534  M=0.628
  ('cnn', 'annotated_old', 'nosus', False)                      AUC=0.894  B=0.794  M=0.853
  ('cnn', 'annotated_old', 'nosus', True)                       AUC=0.937  B=0.794  M=0.853
  ('cnn', 'annotated_old', 'sus', False)                        AUC=0.540  B=0.000  M=0.944
  ('cnn', 'annotated_old', 'sus', True)                         AUC=0.486  B=0.000  M=0.976
  ('cnn', 'roi', 'nosus', False)                                AUC=0.764  B=0.720  M=0.685
  ('cnn', 'roi', 'nosus', True)                                 AUC=0.755  B=0.773  M=0.622
  ('cnn', 'roi', 'sus', False)                            

In [3]:
def pct(v):
    return f"{v*100:.1f}%" if v is not None else "—"

def delta(a, b):
    """b - a, formatted with sign."""
    if a is None or b is None:
        return "—"
    d = (b - a) * 100
    return f"{d:+.1f}pp"

def build_table(dataset_a, dataset_b, balanced, results):
    """Build a comparison DataFrame: dataset_a vs dataset_b."""
    rows = []
    for model in ["cnn", "cytofm_head", "cytofm_finetune"]:
        for sus in ["nosus", "sus"]:
            key_a = (model, dataset_a, sus, balanced)
            key_b = (model, dataset_b, sus, balanced)
            ra = results.get(key_a)
            rb = results.get(key_b)

            def get(r, field, idx=None):
                if r is None: return None
                if idx is not None:
                    pc = r.get("per_class", [])
                    return pc[idx] if len(pc) > idx else None
                return r.get(field)

            a_ben = get(ra, None, 0)
            b_ben = get(rb, None, 0)
            a_mal = get(ra, None, 1)
            b_mal = get(rb, None, 1)
            a_auc = get(ra, "auc")
            b_auc = get(rb, "auc")

            n_a = DATASET_SIZES.get((dataset_a, sus), "—")
            n_b = DATASET_SIZES.get((dataset_b, sus), "—")

            rows.append({
                "Model":         MODEL_LABELS[model],
                "Sus":           "Yes" if sus == "sus" else "No",
                f"N ({DATASET_LABELS[dataset_a]})": n_a,
                f"N ({DATASET_LABELS[dataset_b]})": n_b,
                f"{DATASET_LABELS[dataset_a]} Benign": pct(a_ben),
                f"{DATASET_LABELS[dataset_b]} Benign": pct(b_ben),
                "Benign Δ":      delta(a_ben, b_ben),
                f"{DATASET_LABELS[dataset_a]} Malignant": pct(a_mal),
                f"{DATASET_LABELS[dataset_b]} Malignant": pct(b_mal),
                "Malignant Δ":   delta(a_mal, b_mal),
                f"{DATASET_LABELS[dataset_a]} AUC": pct(a_auc),
                f"{DATASET_LABELS[dataset_b]} AUC": pct(b_auc),
                "AUC Δ":         delta(a_auc, b_auc),
            })
    return pd.DataFrame(rows)

## Table 1 — Round #1 vs Round #2 (Unbalanced)

In [4]:
t1 = build_table("annotated_old", "annotated", balanced=False, results=results)
t1.style.set_caption("Table 1: Round #1 vs Round #2 — Unbalanced Training")

,Model,Sus,N (Round #1),N (Round #2),Round #1 Benign,Round #2 Benign,Benign Δ,Round #1 Malignant,Round #2 Malignant,Malignant Δ,Round #1 AUC,Round #2 AUC,AUC Δ
0,Baseline CNN,No,667,2970,79.4%,39.1%,-40.3pp,85.3%,77.5%,-7.8pp,89.4%,59.6%,-29.8pp
1,Baseline CNN,Yes,817,4802,0.0%,51.5%,+51.5pp,94.4%,67.2%,-27.2pp,54.0%,63.1%,+9.1pp
2,CytoFM (head only),No,667,2970,76.2%,66.0%,-10.2pp,84.3%,63.7%,-20.6pp,91.2%,70.4%,-20.8pp
3,CytoFM (head only),Yes,817,4802,86.2%,92.6%,+6.5pp,67.2%,15.6%,-51.6pp,83.9%,66.6%,-17.4pp
4,CytoFM (fine-tune),No,667,2970,95.2%,77.1%,-18.1pp,90.2%,60.6%,-29.5pp,97.0%,75.6%,-21.4pp
5,CytoFM (fine-tune),Yes,817,4802,80.0%,63.2%,-16.8pp,81.6%,67.2%,-14.4pp,87.6%,73.2%,-14.3pp


## Table 2 — Round #1 vs Round #2 (Balanced)

In [5]:
t2 = build_table("annotated_old", "annotated", balanced=True, results=results)
t2.style.set_caption("Table 2: Round #1 vs Round #2 — Balanced Training")

,Model,Sus,N (Round #1),N (Round #2),Round #1 Benign,Round #2 Benign,Benign Δ,Round #1 Malignant,Round #2 Malignant,Malignant Δ,Round #1 AUC,Round #2 AUC,AUC Δ
0,Baseline CNN,No,667,2970,79.4%,35.0%,-44.3pp,85.3%,77.4%,-7.9pp,93.7%,56.2%,-37.5pp
1,Baseline CNN,Yes,817,4802,0.0%,53.4%,+53.4pp,97.6%,62.8%,-34.8pp,48.6%,61.0%,+12.3pp
2,CytoFM (head only),No,667,2970,65.1%,45.8%,-19.3pp,93.1%,83.2%,-10.0pp,90.6%,71.6%,-19.0pp
3,CytoFM (head only),Yes,817,4802,58.5%,93.6%,+35.2pp,84.0%,12.5%,-71.5pp,83.0%,67.9%,-15.1pp
4,CytoFM (fine-tune),No,667,2970,96.8%,56.9%,-39.9pp,85.3%,75.9%,-9.4pp,97.3%,73.0%,-24.3pp
5,CytoFM (fine-tune),Yes,817,4802,52.3%,89.2%,+36.9pp,95.2%,51.7%,-43.5pp,88.4%,76.0%,-12.4pp


## Table 3 — Round #2 vs ROI (Unbalanced)

In [6]:
t3 = build_table("annotated", "roi", balanced=False, results=results)
t3.style.set_caption("Table 3: Round #2 vs ROI — Unbalanced Training")

,Model,Sus,N (Round #2),N (ROI),Round #2 Benign,ROI Benign,Benign Δ,Round #2 Malignant,ROI Malignant,Malignant Δ,Round #2 AUC,ROI AUC,AUC Δ
0,Baseline CNN,No,2970,2010,39.1%,72.0%,+33.0pp,77.5%,68.5%,-9.0pp,59.6%,76.4%,+16.8pp
1,Baseline CNN,Yes,4802,3410,51.5%,61.5%,+10.1pp,67.2%,69.9%,+2.7pp,63.1%,69.5%,+6.4pp
2,CytoFM (head only),No,2970,2010,66.0%,14.3%,-51.7pp,63.7%,98.0%,+34.2pp,70.4%,81.0%,+10.7pp
3,CytoFM (head only),Yes,4802,3410,92.6%,58.0%,-34.6pp,15.6%,77.7%,+62.0pp,66.6%,71.3%,+4.7pp
4,CytoFM (fine-tune),No,2970,2010,77.1%,73.4%,-3.7pp,60.6%,77.8%,+17.1pp,75.6%,80.2%,+4.7pp
5,CytoFM (fine-tune),Yes,4802,3410,63.2%,49.3%,-13.9pp,67.2%,82.3%,+15.2pp,73.2%,73.2%,-0.0pp


## Table 4 — Round #2 vs ROI (Balanced)

In [7]:
t4 = build_table("annotated", "roi", balanced=True, results=results)
t4.style.set_caption("Table 4: Round #2 vs ROI — Balanced Training")

,Model,Sus,N (Round #2),N (ROI),Round #2 Benign,ROI Benign,Benign Δ,Round #2 Malignant,ROI Malignant,Malignant Δ,Round #2 AUC,ROI AUC,AUC Δ
0,Baseline CNN,No,2970,2010,35.0%,77.3%,+42.3pp,77.4%,62.2%,-15.2pp,56.2%,75.5%,+19.3pp
1,Baseline CNN,Yes,4802,3410,53.4%,74.8%,+21.4pp,62.8%,48.5%,-14.3pp,61.0%,68.6%,+7.6pp
2,CytoFM (head only),No,2970,2010,45.8%,42.7%,-3.1pp,83.2%,89.5%,+6.4pp,71.6%,79.6%,+8.0pp
3,CytoFM (head only),Yes,4802,3410,93.6%,39.5%,-54.1pp,12.5%,87.8%,+75.3pp,67.9%,70.2%,+2.3pp
4,CytoFM (fine-tune),No,2970,2010,56.9%,61.5%,+4.6pp,75.9%,80.1%,+4.2pp,73.0%,80.8%,+7.8pp
5,CytoFM (fine-tune),Yes,4802,3410,89.2%,25.2%,-64.0pp,51.7%,91.1%,+39.4pp,76.0%,71.8%,-4.3pp
